# HybridTokenizer Training Notebook

Trains and saves your custom `HybridTokenizer` on real text corpora.

**What this notebook does:**
1. Feeds text from WikiText-2 + TinyStories into the tokenizer's frequency database  
2. Calls `freeze_vocab()` to finalize base tokens + PMI-scored merge rules  
3. Tests round-trip encode/decode correctness  
4. Saves the tokenizer to `tokenizer.pkl.gz`  

**After running:** copy `tokenizer.pkl.gz` to your Drive or download it — you need it for `slm.ipynb`.

In [1]:
%pip install -q "git+https://github.com/sh20022002/small-Language-Model.git@main"
%pip install -q datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os, time
from datasets import load_dataset
from my_slm.hybrid_tokeniztion import HybridTokenizer

SAVE_PATH = '/content/tokenizer.pkl.gz'   # change if you want Drive storage

## Step 1 — Build the word frequency database

The tokenizer learns which words appear most in your training corpus.  
More diverse text → better vocabulary coverage.  
We use two small datasets that together take ~30 seconds to scan.

In [3]:
print('Loading datasets...')
wiki  = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
wiki  = wiki.filter(lambda x: len(x['text'].strip()) > 20)

# TinyStories: short children stories — great for common English vocabulary
stories = load_dataset('roneneldan/TinyStories', split='train').select(range(50_000))

print(f'WikiText-2 train samples : {len(wiki):,}')
print(f'TinyStories samples      : {len(stories):,}')

Loading datasets...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

...

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

WikiText-2 train samples : 20,887
TinyStories samples      : 50,000


In [4]:
tok = HybridTokenizer(lowercase=True, byte_limit=4)

t0 = time.time()

print('Scanning WikiText-2...')
for ex in wiki:
    tok.add_text(ex['text'])

print('Scanning TinyStories...')
for ex in stories:
    tok.add_text(ex['text'])

elapsed = time.time() - t0
print(f'\nDone in {elapsed:.1f}s  |  unique word types: {len(tok.word_db):,}')

Scanning WikiText-2...
Scanning TinyStories...

Done in 12.2s  |  unique word types: 69,944


In [5]:
# Check coverage before freezing — if base_ratio < 90% consider raising byte_limit
tok.db_status(preview=15, k_bases=2000)

=== HybridTokenizer DB status ===
• Unique word types collected : 69,944
• Total word occurrences       : 10,584,790
• Candidate bases (≤4 bytes) : 9,409  [cover 70.0% of corpus]
• Special tokens reserved      : 9
• Current token2id size        : 9 (includes specials + any byte_* added so far)
• Frozen?                      : False
• Preview of top 15 base candidates:
   the         freq=605115
   and         freq=475270
   to          freq=337096
   a           freq=306792
   was         freq=244490
   he          freq=197984
   she         freq=185699
   it          freq=176199
   they        freq=161858
   her         freq=122072
   in          freq=120996
   of          freq=119764
   with        freq= 87046
   his         freq= 84491
   that        freq= 83960

  < 90 % of corpus occurrences fit base-token criterion.
   Consider raising byte_limit or adding more data before freezing.


## Step 2 — Freeze the vocabulary

Parameters:
- `k_bases`: how many single words to keep as base tokens (top-k by frequency)
- `max_merges`: how many word-pair merges to learn (scored by PMI × log-freq)

Final vocab size ≈ 9 special + 256 byte fallbacks + k_bases + merges (≈ 12 000–13 000)

In [6]:
t0 = time.time()
tok.freeze_vocab(k_bases=2000, max_merges=8000, min_freq=3)
print(f'Frozen in {time.time()-t0:.1f}s')
print(f'Vocab size: {tok.vocab_size:,}')
print(f'  special tokens : {len(tok.special_tokens)}')
print(f'  byte fallbacks : 256')
print(f'  other tokens   : {tok.vocab_size - len(tok.special_tokens) - 256}')

Frozen in 0.3s
Vocab size: 10,265
  special tokens : 9
  byte fallbacks : 256
  other tokens   : 10000


## Step 3 — Verify correctness

In [8]:
# Round-trip test: encode then decode must reproduce original text
# tok.self_test() # Commented out to allow debugging prints to run

# Extra round-trip checks
test_sentences = [
    'The quick brown fox jumps over the lazy dog.',
    'Neural networks learn representations from data.',
    'Hello, world! 🌍  \nNew line here.',
    '1 + 1 = 2  (arithmetic)',
]

print("\n--- Round-trip checks (flat mode) ---")
for s in test_sentences:
    ids   = tok.encode(s, mode='flat')
    back  = tok.decode(ids)
    match = '✓' if back == s else '✗'
    print(f'{match}  [{len(ids):3d} tokens]  (flat) {s[:60]}')
    if back != s:
        print(f'   FAILURE DETECTED FOR FLAT MODE!')
        print(f'   Original : {repr(s)}')
        print(f'   Decoded  : {repr(back)}')

print("\n--- Round-trip checks (rle mode) ---")
for s in test_sentences:
    ids   = tok.encode(s, mode='rle') # Check rle mode, as suggested by traceback
    back  = tok.decode(ids)
    match = '✓' if back == s else '✗'
    print(f'{match}  [{len(ids):3d} tokens]  (rle) {s[:60]}')
    if back != s:
        print(f'   FAILURE DETECTED FOR RLE MODE!')
        print(f'   Original : {repr(s)}')
        print(f'   Decoded  : {repr(back)}')



--- Round-trip checks (flat mode) ---
✗  [ 22 tokens]  (flat) The quick brown fox jumps over the lazy dog.
   FAILURE DETECTED FOR FLAT MODE!
   Original : 'The quick brown fox jumps over the lazy dog.'
   Decoded  : 'the quick brown fox jumps over the lazy dog.'
✗  [ 18 tokens]  (flat) Neural networks learn representations from data.
   FAILURE DETECTED FOR FLAT MODE!
   Original : 'Neural networks learn representations from data.'
   Decoded  : 'neural networks learn representations from data.'
✗  [ 18 tokens]  (flat) Hello, world! 🌍  
New line here.
   FAILURE DETECTED FOR FLAT MODE!
   Original : 'Hello, world! 🌍  \nNew line here.'
   Decoded  : 'hello, world! ���� new line here.'
✗  [ 17 tokens]  (flat) 1 + 1 = 2  (arithmetic)
   FAILURE DETECTED FOR FLAT MODE!
   Original : '1 + 1 = 2  (arithmetic)'
   Decoded  : '1 + 1 = 2 (arithmetic)'

--- Round-trip checks (rle mode) ---
✗  [ 22 tokens]  (rle) The quick brown fox jumps over the lazy dog.
   FAILURE DETECTED FOR RLE MODE!
   

In [9]:
# Inspect how the tokenizer breaks up words
examples = [
    'The capital of France is Paris.',
    'Transformers use self-attention mechanisms.',
    'running runners ran',
    'Hello, world! 🌍  \nNew line here.', # Adding emoji and newline example
    '1 + 1 = 2  (arithmetic)', # Adding whitespace example
]
for text in examples:
    print(f'\nInput: {text}')
    segments = tok.segment(text)
    for token_str, kind in segments:
        print(f'  {token_str:20}  [{kind}]')



Input: The capital of France is Paris.
  the                   [base]
  <SP>                  [sp]
  capit                 [merge]
  al                    [base]
  <SP>                  [sp]
  of                    [base]
  <SP>                  [sp]
  france                [merge]
  <SP>                  [sp]
  is                    [base]
  <SP>                  [sp]
  pa                    [base]
  ri                    [merge]
  s                     [base]
  byte_46               [byte]

Input: Transformers use self-attention mechanisms.
  tran                  [merge]
  s                     [base]
  forme                 [merge]
  rs                    [merge]
  <SP>                  [sp]
  use                   [base]
  <SP>                  [sp]
  self                  [base]
  byte_45               [byte]
  attent                [merge]
  ion                   [merge]
  <SP>                  [sp]
  mech                  [merge]
  ani                   [merge]
  sms          

In [11]:
# Show top learned merges (PMI × frequency)
print('Top 30 learned merges:')
tok.top_merges(30)

Top 30 learned merges:
whenever              = when + ever   (freq≈1262)
backyard              = back + yard   (freq≈1205)
football              = foot + ball   (freq≈695)
yourself              = your + self   (freq≈587)
bathroom              = bath + room   (freq≈533)
thinking              = thin + king   (freq≈417)
baseball              = base + ball   (freq≈379)
careless              = care + less   (freq≈362)
harmless              = harm + less   (freq≈336)
restless              = rest + less   (freq≈322)
helpless              = help + less   (freq≈299)
notebook              = note + book   (freq≈286)
discover              = disc + over   (freq≈276)
backpack              = back + pack   (freq≈260)
wardrobe              = ward + robe   (freq≈229)
passport              = pass + port   (freq≈196)
bookcase              = book + case   (freq≈189)
mustache              = must + ache   (freq≈160)
whatever              = what + ever   (freq≈156)
townsend              = town + send   (freq≈

In [12]:
# Compression ratio: how many tokens per character on average?
sample_text = ' '.join(ex['text'] for ex in wiki.select(range(100)))
ids = tok.encode(sample_text, mode='flat')
chars = len(sample_text)
tokens = len(ids)
print(f'Sample text length : {chars:,} chars')
print(f'Encoded length     : {tokens:,} tokens')
print(f'Compression ratio  : {chars/tokens:.2f} chars/token')
print(f'(BERT averages ~4-5 chars/token; GPT-2 averages ~3-4)')

Sample text length : 48,103 chars
Encoded length     : 23,847 tokens
Compression ratio  : 2.02 chars/token
(BERT averages ~4-5 chars/token; GPT-2 averages ~3-4)


## Step 4 — Save the tokenizer

In [13]:
tok.save(SAVE_PATH)
size_mb = os.path.getsize(SAVE_PATH) / 1e6
print(f'Saved to {SAVE_PATH}  ({size_mb:.1f} MB)')

# Verify: reload and check vocab_size matches
tok2 = HybridTokenizer.load(SAVE_PATH)
assert tok2.vocab_size == tok.vocab_size, 'Vocab size mismatch after reload!'
assert tok2.frozen, 'Tokenizer should be frozen after reload'
print(f'Reload OK  |  vocab_size={tok2.vocab_size:,}')

Saved to /content/tokenizer.pkl.gz  (0.6 MB)
Reload OK  |  vocab_size=10,265


In [ ]:
# Optional: copy to Google Drive so it persists across sessions
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(SAVE_PATH, '/content/drive/MyDrive/slm/tokenizer.pkl.gz')
# print('Copied to Drive')

## How to use the saved tokenizer in `slm.ipynb`

```python
from my_slm.hybrid_tokeniztion import HybridTokenizer

tok = HybridTokenizer.load('/content/tokenizer.pkl.gz')
pad_id    = tok.token2id['<PAD>']
eos_id    = tok.token2id['<EOS>']
vocab_size = tok.vocab_size

# Encode text
ids = tok.encode('Hello world', mode='flat')   # list[int]

# Decode
text = tok.decode(ids)                          # str
```